In [1]:
!pip install pandas
!pip install google_play_scraper
!pip install seaborn matplotlib
!pip install textblob
!pip install wordcloud

In [9]:
import pandas as pd
from google_play_scraper import app
from google_play_scraper import reviews, Sort

apps_review, Token = reviews('id.or.muhammadiyah.quran',
                             country='id',
                             lang='id',
                             count=1000,
                             filter_score_with=None,
                             sort=Sort.MOST_RELEVANT)

# Membuat DataFrame dari data review
df = pd.DataFrame(apps_review)

# Mengonversi kolom timestamp menjadi objek datetime
if 'at' in df.columns:
    df['at'] = pd.to_datetime(df['at'], unit='s')

# Menambahkan filter berdasarkan tanggal
start_date = pd.to_datetime('2023-06-23')
end_date = pd.to_datetime('2024-04-29')

# Memastikan ada kolom 'at' sebelum menambahkan filter
if 'at' in df.columns:
    filtered_df = df[(df['at'] >= start_date) & (df['at'] <= end_date)]
else:
    print("Column 'at' not found in the DataFrame.")

# Memilih kolom 'content' dari DataFrame yang difilter
df_selected = filtered_df['content']

# Menyimpan DataFrame ke dalam file CSV
file_path = r'C:\Users\asus\Downloads\reviewApp.csv'
df_selected.to_csv(file_path, index=False)

print(f"Data review pada aplikasi berhasil disimpan: {file_path}")

Data review pada aplikasi berhasil disimpan: C:\Users\asus\Downloads\reviewApp.csv


In [11]:
import pandas as pd

# Replace 'your_file_path.csv' with the actual path to your CSV file
csv_file_path = r'C:\Users\asus\Downloads\reviewApp.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Display the DataFrame
df

,content
0,Kalau fitur gratis dari sebuah persyarikatan. ...
1,"Sangat mudah digunakan dan mudah dipahami, Sar..."
2,"Alhamdulillah untuk tahap awalnya udah bagus, ..."
3,Fitur pendaftaran sebagai akun Pendidikan keti...
4,Pada bagian memasukkan identitas diri setelah ...
...,...
247,Kerenn
248,Barakallah
249,Bermanfaat
250,Sangat bangga


In [16]:
!pip install Sastrawi

  Obtaining dependency information for Sastrawi from https://files.pythonhosted.org/packages/6f/4b/bab676953da3103003730b8fcdfadbdd20f333d4add10af949dd5c51e6ed/Sastrawi-1.0.1-py2.py3-none-any.whl.metadata
Using cached Sastrawi-1.0.1-py2.py3-none-any.whl (209 kB)


In [18]:
import pandas as pd
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Fungsi case folding
def case_folding(text):
    return text.lower()

# Fungsi tokenisasi
def tokenize(text):
    words = re.findall(r'\b\w+\b', text)
    return words

# Fungsi stemming menggunakan Sastrawi
def stem_text(text):
    stemmer_factory = StemmerFactory()
    stemmer = stemmer_factory.create_stemmer()
    return stemmer.stem(text)

# Fungsi menghapus stopword menggunakan Sastrawi
def remove_stopwords(text):
    stopword_remover_factory = StopWordRemoverFactory()
    stopword_remover = stopword_remover_factory.create_stop_word_remover()
    return stopword_remover.remove(text)

# Fungsi untuk memproses teks secara lengkap
def process_text(text):
    text_lower = case_folding(text)
    tokens = tokenize(text_lower)
    stemmed_text = stem_text(' '.join(tokens))
    text_without_stopwords = remove_stopwords(stemmed_text)
    return text_without_stopwords

# Fungsi untuk memproses kolom pada DataFrame
def process_column(dataframe, column_name):
    dataframe[column_name] = dataframe[column_name].apply(process_text)

# Path file CSV
file_path = r'C:\Users\asus\Downloads\reviewApp.csv'

# Baca DataFrame dari file CSV
df = pd.read_csv(file_path)

# Proses kolom 'Review' untuk case folding, tokenisasi, stemming, dan stopword removal
process_column(df, 'content')

# Simpan DataFrame yang sudah terproses ke dalam file CSV
output_csv_path = r'C:\Users\asus\Downloads\Prepocessing.csv'
df.to_csv(output_csv_path, index=False)

In [21]:
import pandas as pd

# Replace 'your_file_path.csv' with the actual path to your CSV file
csv_file_path = r'C:\Users\asus\Downloads\Prepocessing.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Display the DataFrame
df

,content
0,kalau fitur gratis buah syarikat beuuuhh sanga...
1,sangat mudah mudah paham saran menu dzikir dan...
2,alhamdulillah tahap awal udah bagus masya alla...
3,fitur daftar bagai akun didik di cari menu car...
4,bagi masuk identitas diri unggah foto kta muha...
...,...
247,kerenn
248,barakallah
249,manfaat
250,sangat bangga


In [22]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Kamus kata kunci dan nilai kemiripan
keywords_similarity = {
    'Mudah': ['mudah', 'sederhana', 'simpel', 'simple', 'lancar', 'easy', 'accessible', 'aksesibilitas', 'keterjangkauan', 'mdh', 'ringan', 'gampang', 'mudah diakses', 'mudah digunakan', 'akses mudah', 'navigasi mudah', 'mudahan', 'mudh'],
    'Cepat': ['cepat', 'kilat', 'segera', 'fast', 'cepat mudah', 'cepat tanggap', 'cepat responsif', 'pantas', 'cpt', 'cepat merespon', 'cepat diselesaikan', 'respon cepat', 'penyelesaian cepat', 'cepatan', 'cpat'],
    'Baik': ['baik', 'bagus', 'prima', 'optimal', 'baik sesuai', 'sesuai dengan yang baik', 'kualitas baik', 'kinerja baik', 'good', 'bgus', 'bagus', 'kualitas tinggi', 'layanan baik', 'baikan', 'mantap', 'sesuai', 'cocok', 'tepat', 'pas'],
    'Dipahami': ['dipahami', 'dimengerti', 'dipelajari', 'dipahami', 'understandable', 'mudah dipahami', 'sederhana dipahami', 'jelas dimengerti', 'easy to understand', 'mdh dimengerti', 'easily understood', 'mudah dimengerti', 'jelas dipahami', 'terstruktur dengan baik', 'instruksi yang jelas', 'mudah dipaham'],
    'Efisien': ['efisien', 'efficient', 'efektif', 'hemat', 'produktif', 'efis', 'hemat waktu', 'efisien tinggi', 'waktu penggunaan yang efisien', 'penggunaan sumber daya yang hemat', 'efisiennya', 'tepat waktu hasil', 'hasil yang tepat waktu'],
    'Meningkatkan': ['Meningkatkan', 'enhance', 'peningkatan', 'pembaharuan', 'menaikkan', 'memperbaiki', 'meningkat', 'improve', 'meningkatkan produktivitas', 'meningkatkan efisiensi', 'meningkatkan efektivitas', 'meningkatkan kualitas', 'mgkat'],
    'Stabil': ['stabil', 'tetap', 'kokoh'],
    'Pengguna': ['pengguna', 'penggunaan', 'tampilan', 'penampilan'],
    'Hasil': ['hasil', 'output', 'hasil produksi', 'hasil kerja', 'output', 'hsl', 'hasil akhir', 'output kerja', 'hasil yang diharapkan', 'hasil yang diinginkan', 'hasil-hasil'],
    'Kepuasan': ['kepuasan', 'satisfaction', 'satisfying', 'memenuhi harapan', 'kepuasan', 'kepuasan pengguna', 'kepuasan pelanggan', 'memuaskan', 'memuaskan hati', 'memberikan kepuasan', 'mgkatkan ksnsn', 'memadai', 'meningkatkan puas', 'Meningkatkan kepuasan', 'meningkatkan kepuasan pelanggan', 'kepuasan pengguna yang ditingkatkan', 'meningkatkan kegembiraan', 'memperbaiki kepuasan', 'menaikkan kepuasan', 'improve satisfaction', 'meningkatkan kepuasan pelanggan', 'meningkatkan kualitas layanan', 'memuaskan pelanggan', 'memberikan kepuasan', 'meningkatkan loyalitas', 'meningkatkan retensi pelanggan', 'meningkatkan kepuasan-kepuasan', 'pengalaman pengguna yang memuaskan', 'layanan yang memuaskan', 'kepuasan pelanggan yang berkelanjutan', 'kepuasan pengguna yang tinggi', 'memuaskan-memuaskan', 'memuaskn', 'mgkatkan ksnsn', 'msksn', 'memenuhi', 'puaskan', 'kegembiraan', 'kesenangan', 'kepuasan hati']
}

def preprocess_text(text):
    # Tokenize and lemmatize the text
    tokens = word_tokenize(text.lower())
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Remove stopwords
    stop_words = stopwords.words('english')
    tokens = [token for token in tokens if token not in stop_words]
    
    return ' '.join(tokens)

def text_similarity(text1, text2):
    # Preprocess the texts
    text1 = preprocess_text(text1)
    text2 = preprocess_text(text2)

    # Create the TF-IDF vectors
    vectorizer = TfidfVectorizer()
    vector1 = vectorizer.fit_transform([text1])
    vector2 = vectorizer.transform([text2])

    # Calculate the cosine similarity
    similarity = cosine_similarity(vector1, vector2)

    return similarity[0][0]

# Contoh review
review = """
Kalau fitur gratis dari sebuah persyarikatan  Beuuuhh  sangat mantap ini untuk generasi muda  Kalau berbayar  palingan cuma untuk baca alquran pas di jalan dan fitur jadwal sholat saja  Soalnya fitur kemuhammadiyaan belum ada  Dan kalau seandainya ada fitur langganan majalah Suara Muhammadiyah digital di aplikasi baru worth it  Semoga semakin baik dan bagus fitur fitur nya 
Sangat mudah digunakan dan mudah dipahami  Saran untuk menu dzikir dan tasbih akan lebih baik di tersingkron jadi lebih mudah dalam menggunakan menu dzikir yg di fasilitasi dengan tasbih
Alhamdulillah untuk tahap awalnya udah bagus  Masya Allah  Tapi perlu banyak diperbaiki terutama di bagian Dzikir  Do a Harian  Tasbih  Banyak tulisan  baik tulisan Arab  latin atau terjemahannya  banyak yang kepotong  ada juga yang tidak rapih  Bahkan ada yang  mungkin  dari co pas  jadi nama web nya ikut masuk  Semoga kedepannya bisa diperbaiki lagi dan terus maju  Aamiin   
Fitur pendaftaran sebagai akun Pendidikan ketika di cari menu pencarian nama instansi tidak muncul  hanya muncul   instansi  sehingga tidak bisa melanjutkan pendaftaran  Ketika daftar perorangan tidak bs  jika tidak Menggunakan nomer KTAM  Mungkin bisa diberikan opsi lain tidak harus KTAM  Sehingga publik yg ingin mengakses dapat dengan mudah belajar dari aplikasi yg dimiliki oleh Muhammadiyah 
Pada bagian memasukkan identitas diri setelah unggah foto KTA Muhammadiyah  untuk kecamatan Purwanegara tidak ada desa Merden   Pucungbedug  Desa di kecamatan purwanegara ada     sedangkan di kolom hanya ada     Kalau tidak memasukkan identitas tersebut  tidak bisa masuk ke aplikasi   menggunakan aplikasi  Padahal di awal setelah instal  saya sudah memasukkan nama   NBM  Sebenarnya aplikasinya memiliki fitur  yg bagus  meskipun harus berlangganan dulu untuk menggunakan beberapa fitur yg tersedia
Aplikasi sudah sangat baik namun sayang pada bagian quran tdk bs di zoom yg tentunya menyulitkan dalam membacanya bagi yg memiliki gangguan pengelihatan  plus  minus  silinder  
Aplikasi yg bagus  lebih lengkap dgn mode mushaf  per ayat  perkata ada dan lain lainnya ada  Saran  untuk tulisan arabnya mungkin dipertebal lagi biar lebih jelas bagi yg mata plus dan untuk mushafnya bisa  ketika HPnya di miringkan  mushaf ikut miring juga biar jelas bacanya soslnya tulisan kecil  untuk hiasan bisa di pertipis lagi  Untuk nomor ayat di lquran per ayat ko  tidak masuk ke lingkaran tanda waqaf ya? Terima kasih MU  semoga bermanfaat dan terus berkemajuan
Aplikasinya bagus dan lengkap fiturnya  Mudah mudahan ke depan bisa menjadi super app nya Muhammadiyah  Tolong sediakan juga fitur donasi  zakat  infaq dan shadaqah serta wakaf  bisa kerja sama dengan LAZISMU
QuranMu aplikasi Al Qur an yang sangat membantu saya untuk tetap membaca Al Qur an di manapun ketika saya tidak membawa mushaf teks  fitur ibadah berisi banyak hal terkait ibadah dan hal hal lain yang berguna untuk menambah pengetahuan tentang Islam  terimakasih QuranMu   ¤—
Keren ada banyak fitur fitur dan kitab kitab dan tidak mengambil bacaan dari   bacaan tapi berbagai macamnya  saran kitab hadits nya di tambah ada kitab Arba in Annawawiyah dan disegerakan fitur Muhammadiyah nya
Meski belum mencoba semua menu  tp kami bersyukur  aplikasi ini sedikit menjawab kebutuhan warga Muhammadiyah  mungkin bisa dikembangkan lebih luas  sehingga bisa ada menu berbagi informasi antar pengguna juga  khususnya Informasi kegiatan  secara singkat tentunya
Assalamualaikum min  mohon izin menyampaikan usulan perbaikan untuk adzan subuh dan juga kalender Hijriyah nya seperti nya belum sesuai kriteria Majelis Tarjih yaa  masih yang umum  alangkah baiknya agar disesuaikan sehingga lebih mantap karena ini produk Muhammadiyah  terimakasih   
untuk di menu Iqra saran dan masukan  berikan voice  suara  tiap kata perkata pembacaannya untuk memudahkan umat dalam membaca quran yang baik dan benarnya 
Sudah cukup bagus dengan berbagai fitur yg umum ada di aplikasi lain  namun untuk jadwal adzan sholat jam nya tidak sesuai dengan jadwal yg muhammadiyah  Semoga masih bisa diperbaiki dan makin berkembang 
untuk pengembang aplikasi sabaiknya untuk penampilan awal   Lebih Kepada Inti Islam   seperti JADWAL SAGAT SUSAH UNTUK MENEMUKANNYA    ini menurut saya sebagai generasi muda Muhammadiyah dan yg expert di bidang digital marketing sangat kecewa pengembang aplikasi tidak ini tidak memperhatikan halÂ² inti  karena mayoritas pengguna aplikasi edukasi islam untuk melihat JADWAL Sholat  point kedua  saran saya utk hlm awal terlalu menampakkan ini Apl MU  sehingga terkesan tdk utk umum  Terimakasih    »
Alhamdulillah telah hadir aplikasi QuranMu yang didalamnya tidak hanya berisi Al Qur an dengan berbagai fasilitas seperti kata perkata  tajwid  dll Selain itu ada tambahan menu lain seperti hadits  menu kegiatan ibadah Dan bagi yang ingin belajar membaca  Tahfidz juga disediakan dengan tidak mengenal batasan usia    Semoga aplikasi ini semakin hari semakin banyak yang memanfaatkan dan menu menu lain yang tersedia bisa segera dilengkapi guna menambah khasanah keilmuan yang menggunakan   
Aplikasi sangat lengkap dan bagus  mohon tambahkan bacaan shalat seperti Takbiratul Ikhram  sami  Allahu Liman hamidah  dan keterangan keterangan tiap gerakan  menu nya mungkin bisa ganti nama menjadi sifat sholat nabi
Bagus Aplikasinya  sangat membantu saya terkhusus untuk membaca Qur an  ada juga vitur voice beberapa syekh  jadi bisa juga belajar iramanya
Masih perlu dikembangkan     Notifikasi suara adzan hanya sebentar sekali  tidak sampai selesai    Perlu ditambahkan notifikasi waktu sholat   waktu    Widget belum ada data waktu sholat
Alhamdulillah aplikasi nya mantap dan komplit   Semoga kedepannya bisa di dukung oleh semua warga perserikatan agar bisa berkembang dengan lebih baik lagi    saya berharap kedepannya juga ada aplikasi LAZISMU
Aplikasi bagus  sangat bermanfaat  Cmn saya tidak bsa login krna sya tidak punya KTA  Tolong dibantu pak agar saya punya KTA dan bsa login ke aplikasinya
Aplikasi sangat bagus  utk alquran nya baik yg mushaf maupun per ayat tulisan kurang besar  Sdh diset yg paling besar tetap kurang menurut saya yg penglihatan kabur  Terima kasih
Bagus sekali  tapi notif suara adzan kok tidak ada ya  Tolong developer diperbarui ya biar bisa mengingatkan saat sudah waktunya jam sholat
Secara umum aps ini bagus tetapi jika ditutup harus login kembali dengan memasukkan nomor KTAM  Padahal sudah simpan perubahan  Per    Agust     Logout dari aps harus masuk dg akun pendidikan   
Semoga bermanfaat bagi seluruh warga Muhammadiyah dalam mempelajari Al Qur an  Tolong perbaiki notifikasi adzan  soalnya tidak terdengar suara adzannya  Padahal sudah disetting ada suara adzan 
Alhamdulillah  akhirnya aplikasi khusus untuk warga Muhammadiyah launching juga  Hanya saja untuk menggunakan beberapa fitur di aplikasi wajib punya akun yg datanya diperlukan KTAM  Jadi belum bisa puas pakai aplikasi ini  Tapi apapun itu tetap bintang   buat aplikasi ini
Mohon untuk update nya fitur Muhammadiyah sudah ada min   untuk waktu sholat nya yang subuh belum ikut yg tarjih Muhammadiyah   8 menit   mohon bisa disesuaikan min nanti kalau sesudah update apk nya Jazakumullahu Khairan
Tolong ditambah tuntunan sholat mayyit dan sholatÂ² sunat lain dong min  skaligus saran kalo bisa aplikasinya bs dipakai jg oleh umat tidak terbatas yg pny KTAM saja  Trimksh
Alhamdulillah  tanpa daftar bisa dibuka dengan klik lewati  Bagian doa masih banyak yang perlu diperbaiki  Judul doa  lafad arabnya dan artinya     
Aplikasi yang sangat memuaskan dan memudahkan bagi pengguna  bahkan segala optional berlandaskan dasar yang kuat dan shahih
Aplikasinya sudah bagus  bisa membantu untuk belajar anak  semoga semakin berkembang 
Alhamdulillah  ada aplikasi yang super lengkap dan sangat membantu  khususnya warga Muhammadiyah
Recommended sekali ini dengan banyaknya fitur fitur yang sangat bermanfaat utk memperdalam ilmu agama juga
Sangat bermanfaat sekali  bisa sekalian belajar dan setor ayat  Terimakasih pp muhammadiyah yg sudah buat aplikasi inj
Bagus sudah komplit  Hanya soal pendaftaran yang perlu diperbaiki  Ternyata ada   opsi  selain pakai KTA Muhammadiyah juga bisa pakai akun google 
Aplikasi ini Sangat Bermamfaat Sekali Untuk Membaca dan Menghafal alquran dan Banyak lagi Ilmu Yg bisa didapatkan Terkhusus untuk Warga Muhammadiyah
sangat bermanfaat  sedikit saran pemberitahuan jadwal sholat sering tidak tepat  waktu sholat maghrib tp pemberitahuan asyar    
Alhamdulillah cukup bagus  Akan lebih bagus juga bila logonya berwarna biru kehijauan 
Bagus Untuk daftar saya tidak punya KTA tapi saya input KTP bisa  brarti untuk apa fungsi KTA di form pendaftaran   
Segera dikembangkan kembali  terutama di bagian widget
Alhamdulilaah  terus berlaga meski di gelanggang sunyi  tetap menjadi suluh yang berpijar di setiap kegelapan  Muhammadiyah is the best 
Tidak bisa daftar karena ada beberapa desa yg tidak masuk dlm pilihan  Mohon diperbaiki kembali  Jazzakallah Khair 
Aplikasi paling lengkap untuk kebutuhan muslimin  dan mantapnya iklannya tidak ada   alhamdulillah
Sangat bagus  Lengkap  Satu aplikasi puluhan Fitur  Saya mau melakukan infaq tapi kok gagal terus ya? Mohon solusinya  
Sangat bagus untuk pembelajaran  Fitur sangat lengkap pakai SEKALI Semoga banyak yang terbantu dengan aplikasi ini   
Keren aplikasinya  tapi segera update fitur Muhammadiyah nya biar kita pelajari juga
Sangat bagus dan membantu membaca quran maupun murojaah sendiri ketika belum bisa bertemu guru  Sayangnya al Qurannya belum tersedia yang rasm utsmani dabt madinah  Semoga ke depannya bisa ditambah quran rasm utsmani dabt madinah  Barakallah fiikum 
Mantap banyak fitur yang tak ada di aplikasi lain  saya sebagai anggota mu mengapresiasi atas terbitnya aplikasi Qur anMu  semoga semakin bermanfaat bagi umat dan bangsa kita Aamiin
Aplikasinya bagus  cuma bagi simpatisan Muhammadiyah yang tidak punya KTM bagaimana? Karena pada saat login harus mengisi kolom KTM 
Aplikasi sanggat bagus Tapi kalau waktu sholat kok gak adzan ya  udah saya aktifkan  suaranya
Kok gk ada waktu sholatnya versi Muhammadiyah  blm ada waktu syurqnya jg  Lengkapi min
Sangat inovatif  dan semoga update akan datang bisa menambah fitur seting pada waktu shalat jum at adzan tidak bersuara
Aplikasinya keren  semoga dengan adanya aap ini bisa lebih dekat dengan Al qu an
Sangat bermanfaat fiturÂ²nya lengkap  informasi buat yg tidak ada KTA bisa d isi angka ngacak
Selalu mencerahkan  Semoga aplikasinya selalu bisa di perbaharui
Alhamdulillah ada aplikasi lengkap seperti Quranmu  Bisa belajar dan baca Quran dan kitab hadits dimana saja  semoga kedepan makin update dan memberi banyak manfaat 
aplikasi bagus  namun bagi yang belum memiliki KTA menjadi terhambat mengakses 
Mantab  bisa di tambah untuk yang audio murotalnya 
fitur yg lengkap dan memudahkan  tolong update terkait Menu Muhammadiyah nya
Bagus  Alhamdulillah sebagai warga Muhammadiyah saya bangga karena sudah ada aplikasi lengkap dan khusus yg berguna untuk menunjang khasanah keilmuan
Aplikasinya bagus  fiturnya lengkap  sangat membantu   
Sudah install  melihat fitur fiturnya sangatlah lengkap  Maju teruss
Mungkin aplikasinya bisa dibuat diperuntukan untuk umum  tidak hanya untuk yang memiliki KTAM saja 
Sudah download  Mau daftar gagal terus padahal NBMnya sudah betul
Alhamdulillah  terimakasih sudah menyediakan tempat buat belajar dengan aplikasi ini Muhammadiyah melesat
Bagus sih bagi warga muhammadiyah  tapi ada kurangnya  nggk bisa di dengar secara off line muratalnya
Ini yg ditunggu tunggu  tapi yg tidak memiliki kta tidak bisa daftar
Alhamdulillah sangat membantu Semoga mempermudah dalam beribadah dan membaca Al Qur an
Tolong dong  suara adzan nya gak mau  hanya notif aja yg muncul
Fiturnya lengkap dan sangat membantu
Alhamdulillah   sangat membantu tool legkap dan nyaman dipakai  terimasih Tim Muhammadiyah
Sayangnya nggak ada jadwal Imsakiyahnya  Khususnya pas Ramadan  Padahal  Jadwal Imsyak sangat penting saat Ramadan ini 
Alhamdulillah  semoga aplikasi QuranMu ini tetap terus istiqomah dijalankan dan dikembangkan  Kemudian agar ditambahkan terkait tabligh dan tarjih fatwa Muhammadiyah
Muhammadiyah sudah berpuasa hari ini    Maret       tapi di aplikasi ini baru besok    Maret   Ramadhan  Bagaimana ini kok beda?
Enak  mudah dipakai  tidak berat
harus punya NBM anak anak gak bisa menikmati aplikasi yg bagus ini
Alhamdulillaah    In syaa Allah siap langganan per bulan kalau ditambahkan HPT  Tafsir At Tanwir  tanya jawab agama  dan lain lain  pokoknya semua produk Muhammadiyah deh   
Bagus  lengkap  sangat membantu
Aplikasi keren ga ada iklan
sangat suka fitur fitur nya
Mohon maaf saya harap bisa digunakan secara Universal  karena umat islam nggk semua Muhammadiyah nggk semua Mahasiswa jadi nggk bisa maksimal dalam penggunaan    »
Asallamualaikum Aplikasi bagus  Tapi saya mau login tidak punya KTA  Gmna cara mendapatkannya? Mohon untuk dijawab  
Ikon ibadahnya kurang jreng  Kurang beda antar ikon  Jadi bingung bedainnyaa 
Aplikasinya Sangat Bagus sekali dan sangat bermanfaat untuk umat
sangat bagus Qur anmu  cocok untuk referensi bacaan   kajian
sangat membantu dalam pembelajaran Al Qur an dan ke Islaman
Bintang satu dulu soalnya ribet mau daftar KTA Muhammadiyah yang lama gak bisa digunakan
Awal yg bagus TPI masih perlu peringkat                                                                                                                              
Lancar digunakannya dan ringan
Good Aplikasi    semoga lebih memberi manfaat dan mencerahkan
Bagus  tapi kok waktu jadwal sholat adzannya tidak bunyi ya
Tidak bisa mendaftar karna ga punya KTA  sayang banget 
Bagus mudah di aplikasikan
Mohon informasinya  mau daftar  tapi harus ada KTAM  Jd gak bs masuk 
Bagus aplikasinya  mohon juga di sediakan untuk pengguna Iphone
Sangat suka  tapi kenapa harus berbayar
akhirnya yang di tunggu  launching juga 
Semoga dengan adanya kemudahan belajar lewat aplikasi QuranMu kita semakin cinta Qur an dan dapat mengamalkannya  aamiin
semoga kedepan bisa ada widget waktu sholat ya  baarakallahu fiikum
Sayang tidak punya KTAa jadi tidak bisa daftar
Semoga memberikan manfaat kepada seluruh penggunanya  Maju terus persyarikatan Muhammadiyah 
Catatan      Kalau menyatakan diri sebagai qur anmu   tafsirnya mestinya tafsir at tanwir  Kalau mau menampilkan tafsir lain harus didalam kelompok pilohan tafsir lain     Pengaturan jadwal shalat SAMA SEKALI TIDAK MEMUNCULKAN pengaturan waktu shalat Muhammadiyah
assalamualaikum ayahanda kakanda  gimana cara loginnya kalau belum memiliki KTAM
muhsaf Al Quran nya hurufnya kurang gede tdk bisa di zoom
Alhamdulillah Sangat bermanfaat    izin download   
Dengan mengucapkan syukur alhamdulillah  aplikasi yg bermanfaat
MasyaaAllah aplikasi yg saya nanti dari Muhammadiyah akhirnya ada sangat bagus dan membantu
komplit  bermanfaat untuk umat
Gimana yaa    sudah tf infaq sesuai ketentuan  tp masih belum bisa menggunakan fitur belajar yg dimaksudkan  Apakah saya salah jalur
sangat membantu dan keren
sangat membantu    
Alhamdulillah  sangat luar biasa
Lengkp  tp teks ayat hadis dn tafsir tidak bisa dicopy
Maaf sudah melewati majelis tarjih juga belum ya untuk aplikasi ini
Aplikasinya mantaap betul  Alhamdulillah 
mantap aplikasinya
Alhamdulillah  semoga terus mengabdi untuk mencerahkan umat
fiturnya banyak
Al Qur an  Hadits  Shiroh Nabawiyah  jadwal sholat sampai pengecekan makanan halal ada
Alhamdulilah siip bagus
Saya kasi bintang Mohon izin download aplikasinya ya semoga bermanfaat 
Terus Mencerahkan dan Menyinari Umat dengan Sinar Islam Berkemajuan  hanya tolong di HP INFINIX HOT    Notifikasi dan Suara Adzan Tidak Muncul  tolong perbaikannya Jazakumullah Khoiron Fastabiqul Khoirot
Tingkatkan terus  kitab hadis ditambah lagi
Perlu Perbaikan   update
bagus      teredukasi 
MasyaAlloh    sangat lengkap  dr quran hadist  kisah    rosul  kiblat  sirah nabawiah  Ini superApps muhammadiyah
Sangat membantu dalam belajar Qur an
ini aplikasi yg sangat bagus 
lengkap sekali fiture nya keren
Menjadi amal jariyah  aplikasi yg bermanfaat
Pelafalan kaf sukun pada takbir suara adzan terdengar seperti ada qalqalahnya 
Alhamdulillah    sudah lama di tunggu   Mantap 
Bagus membantu orang orang yg ingin belajar Alquran
Inovasi yg menarik khusus nya bagi warga Muhammadiyaj
Mantab     Sangat bermanfaat
terbaik  memudahkan ibadah
Cara daftar nya gimana belom punya KTA
Bagus sangat bermanfaat    
Aplikasi sangat membantu bagi kaum muslim Muhammadiyah
Alhamdulillah semakin mencerahkan semesta
Berguna  Bermanfaat   Berkemajuan 
mantap  semakin memajukan persyarikatan muhammadiyah
aplikasi yg menjdi kebutuhan umnat  trims muhammadiyah
saya mau daftar ngisi suara muadzin gimana caranya
Alhamdulillah sangat bermanfaat
Sangat berkemajuan
Alhamdulillah     selalu berkemajuan    
Aplikasinya bagus
Alhamdulillah bermanfaat  barokallah hu fie kum
Bagi yang umum loginnya bagaimana
sangat bermanfaat
Sangat direkomendasikan
Aplikasi sangat membantu
Baarakallahu fiik untuk tim pengembang aplikasi QuranMu
Bagus  mendekatkan generasi z pada Qur an
sangat bermanfaat
semakin bermanfaat
bermanfaat sekali
alhamdulillah recomended
sangat bermanfaat
Lanjutkan min  mantap
Aplikasi quran yang lengkap
sangat membantu
Semoga bisa berkembang lagi
Bagus sekali
Tambahkan fitur tafsir min
tolong untuk jadwal subuhnya ditambah 8 menit mengikuti PP Muhammadiyah
cakep pokoknya
Arah kiblatnya tidak sinkron
MasyaAllah  Semoga terus memberi pencerahan buat semesta
Kalender hijriah y tdk sesuai dengan kelender hijriah muhammadiyah
Sebagai alumni santri MBS Muhiba saya bangga sekali dengan Muhammadiyah
MasyaAllah  bagus sekali aplikasinya
Semoga semakin lengkap 
Barokallah  Semoga bermanfaat luas
Semoga dimudahkan
Masya Allah semoga bermanfaat 
Semoga lebih baik akan datang
MasyaAllah bagus bangett
Masya Allah sangat bermanfaat
mencerahkan bangsa
mantap semoga berkah
mantappp appnya
berkemajuan 
bismillah berkah
masya allah bagus sekali lengkap dan insyaallah bermanfaat the best pokoknya apk terbaik yang prnh ak temuin
masya allah bagus bgt aplikasi sangat bermanfaat
sangat bermanfaat jazakumulloh khayr
good aplikasi woow excellent  
aplikasi the best muhammadiyah gerakanku
mantap
bagus
"""

# Menghitung similarity dengan setiap kata kunci dan menampilkan hasilnya
for keyword, similar_words in keywords_similarity.items():
    text2 = ' '.join(similar_words)
    similarity_score = text_similarity(review, text2)
    print(f"Similarity dengan kata kunci '{keyword}': {similarity_score}")


Similarity dengan kata kunci 'Mudah': 0.038201397469062544
Similarity dengan kata kunci 'Cepat': 0.03522699433322129
Similarity dengan kata kunci 'Baik': 0.15112016915950446
Similarity dengan kata kunci 'Dipahami': 0.06757709195792846
Similarity dengan kata kunci 'Efisien': 0.12834984035382271
Similarity dengan kata kunci 'Meningkatkan': 0.0
Similarity dengan kata kunci 'Stabil': 0.024909246573840853
Similarity dengan kata kunci 'Pengguna': 0.01725763225766134
Similarity dengan kata kunci 'Hasil': 0.11624908129963026
Similarity dengan kata kunci 'Kepuasan': 0.09506961363562469
